In [1]:
import nltk
import re
import string
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from google.colab import files
from nltk.corpus import treebank

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('treebank')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('brown')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Unzipping corpora/treebank.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.


True

In [2]:
words = [word.lower() for word in treebank.words()]
vocab = set(words)

In [3]:
def count_word_frequency(words):
    word_count = {}
    for word in words:
        word_count[word] = word_count.get(word, 0) + 1
    return word_count

word_count = count_word_frequency(words)

In [4]:
def calculate_probability(word_count):
    total_words = sum(word_count.values())
    return {word: count / total_words for word, count in word_count.items()}

probabilities = calculate_probability(word_count)

In [5]:
lemmatizer = WordNetLemmatizer()

def lemmatize_word(word):
    """Lemmatize a given word using NLTK WordNet Lemmatizer."""
    return lemmatizer.lemmatize(word)

def delete_letter(word):
    return [word[:i] + word[i+1:] for i in range(len(word))]

def swap_letters(word):
    return [word[:i] + word[i+1] + word[i] + word[i+2:] for i in range(len(word)-1)]

def replace_letter(word):
    letters = string.ascii_lowercase
    return [word[:i] + l + word[i+1:] for i in range(len(word)) for l in letters]

def insert_letter(word):
    letters = string.ascii_lowercase
    return [word[:i] + l + word[i:] for i in range(len(word)+1) for l in letters]

In [6]:
def generate_candidates(word):
    # Use sym_spell.lookup for edit distance 1 candidates
    suggestions = sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=1)
    return {suggestion.term for suggestion in suggestions}

def generate_candidates_level2(word):
    # Use sym_spell.lookup for edit distance 2 candidates
    suggestions = sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
    return {suggestion.term for suggestion in suggestions}

In [7]:
import nltk
from collections import defaultdict

def get_bigram_probability(prev_word, current_word, bigram_frequencies, unigram_probabilities, alpha=0.5):
    """Calculates the combined unigram and bigram probability of a word given the previous word."""
    # Add a smoothing factor (e.g., Laplace smoothing) to avoid zero probabilities for unseen bigrams.
    # For simplicity, we'll use a small constant for unseen bigrams for now, or rely on alpha to balance.

    # Get the bigram frequency
    bigram_freq = bigram_frequencies.get((prev_word, current_word), 0)

    # Get the unigram frequency of the previous word to calculate conditional probability P(current_word | prev_word)
    prev_word_count = word_count.get(prev_word, 0)

    # Calculate bigram probability P(current_word | prev_word)
    # Add 1 for Laplace smoothing in the numerator and denominator
    if prev_word_count > 0:
        # bigram_prob = (bigram_freq + 1) / (prev_word_count + len(vocab)) # Laplace smoothing
        bigram_prob = bigram_freq / prev_word_count # Simple bigram probability without smoothing for now
    else:
        bigram_prob = 0

    # Get the unigram probability of the current word
    unigram_prob = unigram_probabilities.get(current_word, 0)

    # Combine unigram and bigram probabilities using linear interpolation
    # This balances between the more specific bigram context and the more robust unigram probability
    combined_prob = (alpha * bigram_prob) + ((1 - alpha) * unigram_prob)
    return combined_prob

def get_best_correction(word, probs, vocab, max_suggestions=3, prev_word=None):
    candidates_set = set()

    # If the word is already in the vocabulary, it's a primary candidate
    if word in vocab:
        candidates_set.add(word)
    else:
        # Generate candidates with edit distance 1
        level1_candidates = generate_candidates(word)
        candidates_set.update(level1_candidates)

        # If no level 1 candidates, try edit distance 2
        if not candidates_set:
            level2_candidates = generate_candidates_level2(word)
            candidates_set.update(level2_candidates)

    # If no valid candidates found, return an empty list
    if not candidates_set:
        return []

    scored_candidates = []
    for candidate_word in candidates_set:
        unigram_prob = probs.get(candidate_word, 0)
        score = unigram_prob # Default score is unigram probability

        # Incorporate bigram probability if a previous word is available and candidate has unigram prob
        if prev_word and unigram_prob > 0:
            # Use the newly defined get_bigram_probability
            bigram_prob = get_bigram_probability(prev_word, candidate_word, bigram_frequencies, probabilities)

            # Combine unigram and bigram probabilities using a simple linear interpolation
            # Alpha (0.5) is a weighting factor; could be tuned
            alpha = 0.5
            score = alpha * bigram_prob + (1 - alpha) * unigram_prob
            # If bigram_prob is 0, the score will heavily rely on unigram_prob due to (1-alpha) factor.
            # If both are 0, score remains 0.

        scored_candidates.append((candidate_word, score))

    # Sort candidates by their combined score in descending order and return top suggestions
    return sorted(scored_candidates, key=lambda x: x[1], reverse=True)[:max_suggestions]

In [8]:
def correct_sentence(sentence, probabilities, vocab, max_suggestions=1):
    words = word_tokenize(sentence)
    corrected_words = []
    prev_word = None # Initialize prev_word to None

    for i, word in enumerate(words):
        current_word_lower = word.lower()

        if word.isalpha():
            if current_word_lower in vocab:
                corrected_words.append(word)
                prev_word = current_word_lower

            elif word[0].isupper() and current_word_lower not in vocab:
                # If it's a capitalized word and not in vocab, but likely a proper noun
                # or sentence start, keep it as is, but still set as prev_word for context
                corrected_words.append(word)
                prev_word = current_word_lower # still consider it for the next bigram

            else:
                # Pass prev_word to get_best_correction for contextual scoring
                suggestions = get_best_correction(current_word_lower, probabilities, vocab, max_suggestions, prev_word)

                corrected_word_candidate = suggestions[0][0] if suggestions else current_word_lower

                # Preserve original capitalization if the corrected word exists
                if word[0].isupper() and corrected_word_candidate:
                    corrected_word_candidate = corrected_word_candidate.capitalize()

                corrected_words.append(corrected_word_candidate)
                prev_word = corrected_word_candidate.lower() # Update prev_word with the corrected word
        else:
            corrected_words.append(word)
            # If it's not an alphabetic word (punctuation, number, etc.), it doesn't form a bigram context
            # so we reset or keep prev_word as None or handle based on specific needs.
            # For now, let's reset to None for simplicity regarding bigram context.
            prev_word = None # Reset prev_word as punctuation usually breaks word context for simple bigrams

    return ' '.join(corrected_words)

In [9]:
import sys
!{sys.executable} -m pip install symspellpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 12.6 MB/s eta 0:00:00


In [10]:
from symspellpy import SymSpell, Verbosity

sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)

# Populate SymSpell with the word frequencies
for word, count in word_count.items():
    sym_spell.create_dictionary_entry(word, count)

In [11]:
def count_bigram_frequencies(words):
    bigram_counts = defaultdict(int)
    for i in range(len(words) - 1):
        bigram_counts[(words[i], words[i+1])] += 1
    return dict(bigram_counts)

bigram_frequencies = count_bigram_frequencies(words)

In [12]:
user_input_sentence = input("\n Enter a sentence for autocorrection: ")
corrected_sentence = correct_sentence(user_input_sentence, probabilities, vocab)

print("\n Corrected sentence:")
print(corrected_sentence)


 Enter a sentence for autocorrection: My nume is Shrajal

 Corrected sentence:
My name is Shrajal


In [15]:
# Install Gradio (if not installed)
!pip install gradio

import gradio as gr

# Wrapper function for GUI
def autocorrect_text(user_input):
    corrected = correct_sentence(user_input, probabilities, vocab)
    return corrected


# Create Gradio Interface
interface = gr.Interface(
    fn=autocorrect_text,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Enter a sentence with spelling mistakes...",
        label="Input Sentence"
    ),
    outputs=gr.Textbox(
        label="Corrected Sentence"
    ),
    title="🧠 NLP Auto-Correction System",
    description="This system corrects spelling mistakes and give the correct sentences."
)

# Launch the app
interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://99bf3c97d7b9e44979.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
